# Polymarket Connection Checks

This notebook is reserved for Milestone 1 connectivity and payload validation.

Goals:
- verify public REST connectivity
- inspect payload shape and stable identifiers
- test WebSocket subscription flow
- record sample payloads for later schema design


## Planned checks

1. Verify Gamma `GET /markets` reachability and capture a representative market sample
2. Verify public CLOB `GET /book`, `GET /price`, and `GET /prices-history` access for one Gamma token
3. Confirm the key join identifiers shared across Gamma and CLOB payloads
4. Defer Data API verification to T1.2 and WebSocket verification to T1.3
5. Record local sample payload locations for later schema design


In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import httpx
from dotenv import load_dotenv


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TASKS.md").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


load_dotenv()

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEFAULT_BASE_URLS = {
    "gamma": "https://gamma-api.polymarket.com",
    "clob": "https://clob.polymarket.com",
}
BASE_URLS = {
    "gamma": os.getenv("POLYMARKET_GAMMA_BASE_URL", DEFAULT_BASE_URLS["gamma"]),
    "clob": os.getenv("POLYMARKET_CLOB_BASE_URL", DEFAULT_BASE_URLS["clob"]),
}
SAMPLE_DIRS = {
    "gamma": REPO_ROOT / "data/raw/gamma/connection_checks",
    "clob": REPO_ROOT / "data/raw/clob/connection_checks",
}

for sample_dir in SAMPLE_DIRS.values():
    sample_dir.mkdir(parents=True, exist_ok=True)


def fetch_json(url: str, params: dict[str, Any] | None = None) -> Any:
    response = httpx.get(
        url,
        params=params,
        headers={"Accept": "application/json"},
        timeout=20.0,
    )
    response.raise_for_status()
    return response.json()


def parse_clob_token_ids(raw_value: Any) -> list[str]:
    if raw_value is None:
        return []
    if isinstance(raw_value, list):
        return [str(item) for item in raw_value if str(item).strip()]
    if isinstance(raw_value, str):
        stripped = raw_value.strip()
        if not stripped:
            return []
        try:
            decoded = json.loads(stripped)
        except json.JSONDecodeError:
            return [part.strip() for part in stripped.split(",") if part.strip()]
        if isinstance(decoded, list):
            return [str(item) for item in decoded if str(item).strip()]
    raise ValueError(f"Unsupported clobTokenIds value: {raw_value!r}")


def slugify_endpoint(endpoint: str) -> str:
    return endpoint.strip("/").replace("/", "_").replace("-", "_") or "root"


def save_sample_payload(source: str, endpoint: str, params: dict[str, Any] | None, payload: Any) -> Path:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    wrapped_payload = {
        "source": source,
        "endpoint": endpoint,
        "params": params or {},
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
        "payload": payload,
    }
    sample_path = SAMPLE_DIRS[source] / f"{timestamp}_{slugify_endpoint(endpoint)}.json"
    sample_path.write_text(json.dumps(wrapped_payload, indent=2, sort_keys=True))
    return sample_path


def format_market_identifiers(market: dict[str, Any]) -> dict[str, Any]:
    return {
        "market_id": market.get("id"),
        "question": market.get("question"),
        "slug": market.get("slug"),
        "conditionId": market.get("conditionId") or market.get("condition_id"),
        "clobTokenIds": parse_clob_token_ids(market.get("clobTokenIds")),
    }


print("Base URLs:")
for source_name, base_url in BASE_URLS.items():
    print(f"- {source_name}: {base_url}")

print("\nSample directories:")
for source_name, sample_dir in SAMPLE_DIRS.items():
    print(f"- {source_name}: {sample_dir}")


In [ ]:
gamma_endpoint = "/markets"
gamma_params = {"limit": 10, "closed": False}
gamma_response = fetch_json(f"{BASE_URLS['gamma']}{gamma_endpoint}", params=gamma_params)
gamma_sample_path = save_sample_payload("gamma", gamma_endpoint, gamma_params, gamma_response)

if isinstance(gamma_response, dict) and "data" in gamma_response:
    gamma_markets = gamma_response["data"]
else:
    gamma_markets = gamma_response

if not isinstance(gamma_markets, list):
    raise TypeError(f"Expected the Gamma /markets response to be a list, got {type(gamma_markets)!r}")

usable_gamma_market = None
for market in gamma_markets:
    identifiers = format_market_identifiers(market)
    if identifiers["conditionId"] and identifiers["clobTokenIds"]:
        usable_gamma_market = market
        gamma_market_identifiers = identifiers
        break

if usable_gamma_market is None:
    raise RuntimeError("No Gamma market in the sample response exposed both conditionId and clobTokenIds.")

print("Selected Gamma market identifiers:")
print(json.dumps(gamma_market_identifiers, indent=2))
print(f"Saved Gamma sample: {gamma_sample_path}")


In [ ]:
if "gamma_market_identifiers" not in globals():
    raise RuntimeError("Run the Gamma market cell before the CLOB verification cell.")

condition_id = str(gamma_market_identifiers["conditionId"])
token_id = str(gamma_market_identifiers["clobTokenIds"][0])

book_endpoint = "/book"
book_params = {"token_id": token_id}
book_payload = fetch_json(f"{BASE_URLS['clob']}{book_endpoint}", params=book_params)
book_sample_path = save_sample_payload("clob", book_endpoint, book_params, book_payload)

buy_price_endpoint = "/price"
buy_price_params = {"token_id": token_id, "side": "BUY"}
buy_price_payload = fetch_json(f"{BASE_URLS['clob']}{buy_price_endpoint}", params=buy_price_params)
buy_price_sample_path = save_sample_payload("clob", buy_price_endpoint, buy_price_params, buy_price_payload)

sell_price_endpoint = "/price"
sell_price_params = {"token_id": token_id, "side": "SELL"}
sell_price_payload = fetch_json(f"{BASE_URLS['clob']}{sell_price_endpoint}", params=sell_price_params)
sell_price_sample_path = save_sample_payload("clob", sell_price_endpoint, sell_price_params, sell_price_payload)

history_endpoint = "/prices-history"
history_params = {"market": token_id, "interval": "1w", "fidelity": 5}
history_payload = fetch_json(f"{BASE_URLS['clob']}{history_endpoint}", params=history_params)
history_sample_path = save_sample_payload("clob", history_endpoint, history_params, history_payload)

book_market = str(book_payload.get("market"))
book_asset_id = str(book_payload.get("asset_id"))
condition_matches = condition_id == book_market
token_matches = token_id == book_asset_id

print("Selected Gamma and CLOB join keys:")
print(
    json.dumps(
        {
            "Gamma.conditionId": condition_id,
            "Gamma.clobTokenIds[0]": token_id,
            "CLOB book.market": book_market,
            "CLOB book.asset_id": book_asset_id,
        },
        indent=2,
    )
)
print(f"Gamma.conditionId == CLOB book.market: {condition_matches}")
print(f"Gamma.clobTokenIds[0] == CLOB book.asset_id: {token_matches}")

if not condition_matches or not token_matches:
    raise AssertionError("Gamma and CLOB identifiers did not line up as expected.")

history_points = history_payload.get("history") if isinstance(history_payload, dict) else history_payload
history_count = len(history_points) if isinstance(history_points, list) else "unknown"

print("\nSaved CLOB samples:")
for sample_path in [book_sample_path, buy_price_sample_path, sell_price_sample_path, history_sample_path]:
    print(f"- {sample_path}")

print("\nRepresentative payload notes:")
print(f"- buy price payload keys: {sorted(buy_price_payload.keys()) if isinstance(buy_price_payload, dict) else 'n/a'}")
print(f"- sell price payload keys: {sorted(sell_price_payload.keys()) if isinstance(sell_price_payload, dict) else 'n/a'}")
print(f"- prices-history point count: {history_count}")
print("- note: the live CLOB API currently requires fidelity >= 5 for a 1w history query")


In [ ]:
print("Deferred to T1.2: Data API wallet, holder, trade-history, and open-interest endpoint checks live in the next task.")


In [ ]:
print("Deferred to T1.3: WebSocket connectivity and message-shape verification are intentionally excluded from T1.1.")


## Findings template

Update this section after running the notebook locally.

- Reachable endpoints:
  - Gamma `GET /markets`
  - CLOB `GET /book`
  - CLOB `GET /price` for both `BUY` and `SELL`
  - CLOB `GET /prices-history` using `market=<token_id>`, `interval=1w`, and `fidelity=5`
- Required auth or headers:
  - Note whether public access was sufficient.
- Stable market or token identifiers:
  - `Gamma.conditionId`
  - `Gamma.clobTokenIds[0]`
  - `CLOB book.market`
  - `CLOB book.asset_id`
- Saved sample locations:
  - `data/raw/gamma/connection_checks/`
  - `data/raw/clob/connection_checks/`
- Missing data or blockers:
